# EDA - Contents Master

Exploratory analysis of the contents policies master file.

In [0]:
%pip install openpyxl seaborn

In [0]:
df_contents = spark.read.parquet(
    "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/workings/policies_contents_master.parquet"
)
display(df_contents.limit(10))

In [0]:
# Trim to a smaller dataframe of features currently being used plus target and do EDA on that
# Year is derived in the next cell and appended there; period_start/period_end are kept
# in the select (not the variable list) because the Year cell derives from them.
variables_contents = [
    "exposure", "policyholder_age", "sum_insured", "tenure", "security_level",
    "high_value_items", "accidental_damage_cover", "works_from_home", "lodgers", "bicycle_cover",
    "voluntary_excess", "compulsory_excess", "channel", "payment_method", "missed_payments_12m",
    "new_business", "renewal_count", "days_quote_to_inception", "hail_days", "wind_exposure_index",
    "freezing_days", "dry_summer_index", "wildfire_index", "rainfall_mm", "mean_temperature_c",
    "occupation", "marital_status", "household_size", "credit_band", "prior_claims_5y",
    "licence_points", "years_at_address", "branch_id", "marketing_segment", "region",
    "urban_density", "flood_zone", "distance_to_river_km", "elevation_m", "soil_type",
    "year_built", "property_type", "roof_type", "basement", "trees_within_5m",
    "listed_building", "bedrooms", "energy_rating", "solar_panels", "ultimate_claims",
]

df_contents_trim = df_contents.select(
    "address_id",
    "customer_id",
    "row_id",
    "policy_id",
    "period_start",
    "period_end",
    "earned_premium",
    *variables_contents,
).toPandas()
display(df_contents_trim.head(10))

In [0]:
import pandas as pd

df_contents_trim["Year"] = pd.to_datetime(df_contents_trim["period_start"]).dt.year
variables_contents.append("Year")

In [0]:
categorical_contents = [
    c for c in variables_contents
    if c not in ("period_start", "period_end")
    and (df_contents_trim[c].dtype in ("object", "bool") or c == "Year")
]

counts = {c: df_contents_trim[c].value_counts(dropna=False) for c in categorical_contents}

for c, vc in counts.items():
    print(f"--- {c} ---")
    print(vc)

excel_path = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/Excel files/Contents_EDA.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for c, vc in counts.items():
        vc.rename("count").rename_axis(c).to_excel(writer, sheet_name=c)

print(f"Saved value counts to {excel_path}")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

numeric_contents = [
    c for c in variables_contents
    if c != "Year" and pd.api.types.is_numeric_dtype(df_contents_trim[c])
]

for c in numeric_contents:
    sns.boxplot(data=df_contents_trim, y=c)
    plt.title(c)
    plt.show()

In [0]:
import io
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage

excel_path = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/Excel files/Contents_EDA.xlsx"

numeric_contents = [
    c for c in variables_contents
    if c != "Year" and pd.api.types.is_numeric_dtype(df_contents_trim[c])
]

wb = load_workbook(excel_path)

for c in numeric_contents:
    fig, ax = plt.subplots()
    sns.boxplot(data=df_contents_trim, y=c, ax=ax)
    ax.set_title(c)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    plt.close(fig)
    buf.seek(0)
    img = XLImage(buf)
    ws = wb.create_sheet(f"{c}_boxplot"[:31])
    ws.add_image(img, "B2")

wb.save(excel_path)
print(f"Added {len(numeric_contents)} boxplots to {excel_path}")

In [0]:
over_one = df_contents_trim[df_contents_trim["exposure"] > 1]

print(f"Rows with exposure > 1: {len(over_one):,} of {len(df_contents_trim):,} ({len(over_one) / len(df_contents_trim):.2%})")
print(over_one["exposure"].describe())

excel_path = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/Excel files/Contents_EDA.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    over_one.to_excel(writer, sheet_name="exposure_over_1", index=False)

print(f"Saved {len(over_one):,} rows to {excel_path} (sheet: exposure_over_1)")